# 节点18：Stable Diffusion — 隐空间扩散模型（2022）

**论文**：Rombach et al., "High-Resolution Image Synthesis with Latent Diffusion Models", CVPR 2022, arXiv:2112.10752

本 notebook 用纯 NumPy 演示 Latent Diffusion 的核心思想：
1. 自编码器：压缩与还原
2. 隐空间 DDIM 采样
3. 文本条件控制
4. 像素空间 vs 隐空间计算量对比


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ============================================================
# 常量定义
# ============================================================
PIXEL_DIM = 256     # 原始"图片"维度（简化为向量）
LATENT_DIM = 16     # 隐空间维度（压缩比 = 256/16 = 16x）
TEXT_DIM = 32       # 文字向量维度
T = 100             # 扩散步数
BETA_START = 1e-4
BETA_END = 0.02

print(f"压缩比: {PIXEL_DIM}/{LATENT_DIM} = {PIXEL_DIM//LATENT_DIM}x")
print(f"像素维度: {PIXEL_DIM}")
print(f"隐空间维度: {LATENT_DIM}")
print(f"扩散步数: {T}")


## Part 1：线性自编码器

最简单的自编码器：**编码器 = 线性矩阵压缩，解码器 = 线性矩阵还原**。


In [ ]:
class LinearAutoencoder:
    """线性自编码器：展示压缩与还原的基本原理"""

    def __init__(self, pixel_dim, latent_dim, lr=0.01):
        self.pixel_dim = pixel_dim
        self.latent_dim = latent_dim
        self.lr = lr
        # 编码器权重：pixel_dim → latent_dim
        self.W_enc = np.random.randn(latent_dim, pixel_dim) * 0.001
        # 解码器权重：latent_dim → pixel_dim
        self.W_dec = np.random.randn(pixel_dim, latent_dim) * 0.001

    def encode(self, x):
        """压缩：高维 → 低维"""
        h = np.clip(self.W_enc @ x, -30, 30)
        return np.tanh(h)

    def decode(self, z):
        """还原：低维 → 高维"""
        h = np.clip(self.W_dec @ z, -30, 30)
        return np.tanh(h)

    def reconstruct(self, x):
        """完整前向：压缩 → 还原"""
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def train_step(self, x):
        """一步梯度更新（带梯度裁剪防溢出）"""
        z = self.encode(x)
        x_hat = self.decode(z)
        loss = np.mean((x_hat - x) ** 2)

        # 反向传播（链式法则）
        grad_xhat = 2 * (x_hat - x) / self.pixel_dim
        pre_dec = np.clip(self.W_dec @ z, -30, 30)
        grad_z_dec = (1 - np.tanh(pre_dec) ** 2) * grad_xhat
        grad_Wdec = np.outer(grad_z_dec, z)
        pre_enc = np.clip(self.W_enc @ x, -30, 30)
        grad_z_enc = self.W_dec.T @ grad_z_dec
        grad_z_pre = (1 - np.tanh(pre_enc) ** 2) * grad_z_enc
        grad_Wenc = np.outer(grad_z_pre, x)

        # 梯度裁剪，防止 exploding gradient
        grad_Wdec = np.clip(grad_Wdec, -1.0, 1.0)
        grad_Wenc = np.clip(grad_Wenc, -1.0, 1.0)

        self.W_dec -= self.lr * grad_Wdec
        self.W_enc -= self.lr * grad_Wenc
        return loss


# 生成合成"图片"数据集（正弦波模拟结构性数据）
def make_synthetic_images(n, pixel_dim):
    """生成有结构的合成数据（正弦波+随机偏移，比纯随机更像真实图片）"""
    imgs = []
    for _ in range(n):
        t = np.linspace(0, 2 * np.pi, pixel_dim)
        freq = np.random.uniform(0.5, 3.0)
        phase = np.random.uniform(0, 2 * np.pi)
        amplitude = np.random.uniform(0.3, 1.0)
        img = amplitude * np.sin(freq * t + phase)
        # 添加少量噪声
        img += np.random.randn(pixel_dim) * 0.05
        imgs.append(img)
    return np.array(imgs)

dataset = make_synthetic_images(200, PIXEL_DIM)
print(f"数据集形状: {dataset.shape}")

# 训练自编码器
ae = LinearAutoencoder(PIXEL_DIM, LATENT_DIM, lr=0.005)
losses = []
for epoch in range(300):
    epoch_loss = 0
    for x in dataset:
        epoch_loss += ae.train_step(x)
    losses.append(epoch_loss / len(dataset))

# 测试重建质量
test_img = dataset[0]
x_hat, z = ae.reconstruct(test_img)
recon_mse = np.mean((x_hat - test_img) ** 2)
print(f"重建 MSE（训练后）: {recon_mse:.4f}")
print(f"隐向量形状: {z.shape}（压缩了 {PIXEL_DIM//LATENT_DIM}x）")


In [ ]:
# 可视化训练曲线和重建效果
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. 训练损失曲线
axes[0].plot(losses, color='steelblue', linewidth=2)
axes[0].set_title("自编码器训练损失", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("重建 MSE")
axes[0].grid(alpha=0.3)
axes[0].text(0.6, 0.8, f"最终损失: {losses[-1]:.4f}",
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.5))

# 2. 原图 vs 重建
t = np.linspace(0, 2 * np.pi, PIXEL_DIM)
axes[1].plot(t, test_img, label="原始图片", color="steelblue", linewidth=2)
axes[1].plot(t, x_hat, label="重建图片", color="tomato", linewidth=2, linestyle="--")
axes[1].set_title(f"原图 vs 重建（压缩 {PIXEL_DIM//LATENT_DIM}x）", fontsize=13)
axes[1].set_xlabel("位置")
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3. 隐向量可视化
axes[2].bar(range(LATENT_DIM), z, color="mediumpurple", edgecolor="white")
axes[2].set_title(f"隐向量（{LATENT_DIM} 维）", fontsize=13)
axes[2].set_xlabel("隐向量维度")
axes[2].set_ylabel("值")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../docs/assets/18-autoencoder.png", dpi=100, bbox_inches="tight")
plt.show()
print("图片已保存")


## Part 2：噪声调度（隐空间扩散的基础）

和 DDPM/DDIM 一样，我们需要一个噪声调度表。但这次扩散在**隐空间**里进行。


In [ ]:
# 噪声调度（和节点16/17相同）
betas = np.linspace(BETA_START, BETA_END, T)
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)

# 验证调度性质
assert alpha_bar[0] > 0.99, "t=0时应接近原始数据"
assert alpha_bar[-1] < 0.5, "t=T时应有大量噪声"
assert np.all(np.diff(alpha_bar) < 0), "alpha_bar应单调递减"

print("噪声调度性质验证通过")
print(f"  ᾱ_0  = {alpha_bar[0]:.4f}（接近1，几乎无噪声）")
print(f"  ᾱ_50 = {alpha_bar[50]:.4f}（中间状态）")
print(f"  ᾱ_99 = {alpha_bar[-1]:.4f}（接近0，几乎全噪声）")

# 正向加噪函数（作用于隐向量）
def forward_diffusion(z0, t_step, alpha_bar):
    """把干净隐向量 z0 加噪到第 t_step 步"""
    ab = alpha_bar[t_step]
    eps = np.random.randn(*z0.shape)
    z_t = np.sqrt(ab) * z0 + np.sqrt(1 - ab) * eps
    return z_t, eps

# 测试正向扩散
z_clean = np.random.randn(LATENT_DIM)
z_noisy, _ = forward_diffusion(z_clean, T - 1, alpha_bar)
print(f"\n干净隐向量范数: {np.linalg.norm(z_clean):.3f}")
print(f"加噪隐向量范数: {np.linalg.norm(z_noisy):.3f}")


## Part 3：文本条件控制（Cross-Attention 简化版）

真实 Stable Diffusion 用 CLIP 把文字转成向量，然后通过 Cross-Attention 影响去噪过程。
这里用**简化版**：不同的"文字描述"→不同的 embedding 向量→影响去噪方向。


In [ ]:
class TextConditionedDenoiser:
    """带文本条件的去噪网络（简化线性版本）"""

    def __init__(self, latent_dim, text_dim):
        self.latent_dim = latent_dim
        self.text_dim = text_dim
        # 去噪网络权重（处理噪声隐向量）
        self.W_noise = np.random.randn(latent_dim, latent_dim) * 0.1
        # 文本条件权重（如何把文字信息混入）
        self.W_text = np.random.randn(latent_dim, text_dim) * 0.1

    def predict_noise(self, z_t, text_vec, t_step, T):
        """预测噪声 ε（给定 noisy 隐向量 z_t 和文字向量）"""
        # 时间步编码（正弦编码，简化版）
        t_enc = np.sin(np.pi * t_step / T * np.linspace(0, 1, self.latent_dim))

        # 文本条件通过线性变换混入（Cross-Attention 简化版）
        text_condition = self.W_text @ text_vec

        # 综合：噪声隐向量 + 时间步 + 文本条件
        h = np.tanh(self.W_noise @ z_t + t_enc)
        eps_pred = h + 0.1 * text_condition
        return eps_pred


# 模拟两种不同"文字描述"
text_cat = np.array([1.0] + [0.0] * (TEXT_DIM - 1))    # "橘猫"
text_sky = np.array([0.0, 1.0] + [0.0] * (TEXT_DIM - 2))  # "蓝天"
# 无条件向量（空文字描述）
text_uncond = np.zeros(TEXT_DIM)

denoiser = TextConditionedDenoiser(LATENT_DIM, TEXT_DIM)
z_noise = np.random.randn(LATENT_DIM)

# 验证不同文字条件产生不同预测
pred_cat = denoiser.predict_noise(z_noise, text_cat, 50, T)
pred_sky = denoiser.predict_noise(z_noise, text_sky, 50, T)
pred_unc = denoiser.predict_noise(z_noise, text_uncond, 50, T)

diff_cat_sky = np.linalg.norm(pred_cat - pred_sky)
diff_cat_unc = np.linalg.norm(pred_cat - pred_unc)

print(f"'橘猫' vs '蓝天' 预测差异: {diff_cat_sky:.4f}")
print(f"'橘猫' vs 无条件 预测差异: {diff_cat_unc:.4f}")
assert diff_cat_sky > 0, "不同文字条件应产生不同预测"
print("文字条件控制验证通过：不同描述产生不同噪声预测")


## Part 4：隐空间 DDIM 采样

结合自编码器和条件去噪器，在隐空间里做 DDIM 采样，最后解码还原为"图片"。


In [ ]:
def ddim_step_latent(z_t, t, t_prev, alpha_bar, denoiser, text_vec, eta=0.0):
    """
    DDIM 隐空间采样单步（确定性，eta=0）
    z_t:        当前时间步的 noisy 隐向量
    t, t_prev:  时间步索引
    返回:       z_{t_prev}
    """
    ab_t = alpha_bar[t]
    ab_prev = alpha_bar[t_prev] if t_prev >= 0 else 1.0

    # 预测噪声
    eps_pred = denoiser.predict_noise(z_t, text_vec, t, T)

    # 预测干净隐向量 z_0
    z0_pred = (z_t - np.sqrt(1 - ab_t) * eps_pred) / np.sqrt(ab_t)
    z0_pred = np.clip(z0_pred, -3, 3)

    # DDIM 前进方向
    z_prev = np.sqrt(ab_prev) * z0_pred + np.sqrt(1 - ab_prev) * eps_pred

    # eta > 0 时添加随机性（这里 eta=0 为确定性）
    if eta > 0:
        sigma = eta * np.sqrt((1 - ab_prev) / (1 - ab_t)) * np.sqrt(1 - ab_t / ab_prev)
        z_prev += sigma * np.random.randn(*z_t.shape)

    return z_prev


def latent_ddim_sample(denoiser, text_vec, alpha_bar, n_steps=20):
    """在隐空间里做 DDIM 采样，返回干净隐向量"""
    # 从纯噪声开始
    z = np.random.randn(LATENT_DIM)

    # 均匀选取时间步
    timesteps = np.linspace(T - 1, 0, n_steps + 1, dtype=int)

    for i in range(len(timesteps) - 1):
        t = timesteps[i]
        t_prev = timesteps[i + 1]
        z = ddim_step_latent(z, t, t_prev, alpha_bar, denoiser, text_vec)

    return z


# 生成两种条件下的隐向量
np.random.seed(42)
z_cat = latent_ddim_sample(denoiser, text_cat, alpha_bar, n_steps=20)
np.random.seed(42)
z_sky = latent_ddim_sample(denoiser, text_sky, alpha_bar, n_steps=20)

# 解码：隐向量 → 图片
img_cat = ae.decode(z_cat)
img_sky = ae.decode(z_sky)

print("隐空间采样完成！")
print(f"'橘猫'隐向量范数: {np.linalg.norm(z_cat):.3f}")
print(f"'蓝天'隐向量范数: {np.linalg.norm(z_sky):.3f}")
print(f"两者差异: {np.linalg.norm(z_cat - z_sky):.3f}（不同文字 → 不同隐向量 → 不同图片）")


In [ ]:
# 可视化：文本条件对生成的影响
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

t_axis = np.linspace(0, 2 * np.pi, PIXEL_DIM)

axes[0].plot(t_axis, img_cat, color="darkorange", linewidth=2, label="'橘猫'")
axes[0].plot(t_axis, img_sky, color="steelblue", linewidth=2, label="'蓝天'")
axes[0].set_title("不同文字条件 → 不同生成图片", fontsize=13)
axes[0].set_xlabel("位置")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 隐向量对比
x_bar = np.arange(LATENT_DIM)
width = 0.35
axes[1].bar(x_bar - width/2, z_cat, width, color="darkorange", label="'橘猫'", alpha=0.8)
axes[1].bar(x_bar + width/2, z_sky, width, color="steelblue", label="'蓝天'", alpha=0.8)
axes[1].set_title(f"生成的隐向量（{LATENT_DIM}维）", fontsize=13)
axes[1].set_xlabel("维度")
axes[1].set_ylabel("值")
axes[1].legend()
axes[1].grid(alpha=0.3)

# 步数对比（确定性采样，相同种子）
step_options = [5, 10, 20, 50]
differences = []
for n_steps in step_options:
    np.random.seed(42)
    z_ref = latent_ddim_sample(denoiser, text_cat, alpha_bar, n_steps=50)
    np.random.seed(42)
    z_approx = latent_ddim_sample(denoiser, text_cat, alpha_bar, n_steps=n_steps)
    differences.append(np.linalg.norm(z_ref - z_approx))

axes[2].plot(step_options, differences, 'o-', color="mediumpurple", linewidth=2, markersize=8)
axes[2].set_title("DDIM 步数 vs 与50步的差异", fontsize=13)
axes[2].set_xlabel("采样步数")
axes[2].set_ylabel("与50步结果的差异（L2）")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../docs/assets/18-latent-ddim.png", dpi=100, bbox_inches="tight")
plt.show()
print("图片已保存")


## Part 5：像素空间 vs 隐空间计算量对比

为什么隐空间扩散这么重要？让我们用数字说话。


In [ ]:
import time

def compute_cost_comparison(pixel_dim_list, latent_dim_list, T=100, n_steps=20):
    """比较像素空间和隐空间的扩散计算量"""
    results = []
    for px_dim, lat_dim in zip(pixel_dim_list, latent_dim_list):
        compression = px_dim // lat_dim

        # 模拟像素空间扩散（每步计算量 ∝ 维度的平方，矩阵乘法）
        W_px = np.random.randn(px_dim, px_dim) * 0.01
        start = time.perf_counter()
        for _ in range(n_steps):
            x = np.random.randn(px_dim)
            _ = np.tanh(W_px @ x)
        pixel_time = time.perf_counter() - start

        # 模拟隐空间扩散（每步计算量 ∝ 隐空间维度的平方）
        W_lat = np.random.randn(lat_dim, lat_dim) * 0.01
        start = time.perf_counter()
        for _ in range(n_steps):
            z = np.random.randn(lat_dim)
            _ = np.tanh(W_lat @ z)
        latent_time = time.perf_counter() - start

        results.append({
            "pixel_dim": px_dim,
            "latent_dim": lat_dim,
            "compression": compression,
            "pixel_time_ms": pixel_time * 1000,
            "latent_time_ms": latent_time * 1000,
            "speedup": pixel_time / latent_time if latent_time > 0 else float('inf')
        })
    return results

# 测试不同压缩比
pixel_dims  = [64,  128,  256]
latent_dims = [8,   16,   16]

results = compute_cost_comparison(pixel_dims, latent_dims)

print(f"{'像素维度':>8} {'隐空间维度':>10} {'压缩比':>6} {'像素时间(ms)':>14} {'隐空间时间(ms)':>16} {'加速比':>8}")
print("-" * 75)
for r in results:
    print(f"{r['pixel_dim']:>8d} {r['latent_dim']:>10d} {r['compression']:>6d}x "
          f"{r['pixel_time_ms']:>14.2f} {r['latent_time_ms']:>16.2f} {r['speedup']:>7.1f}x")


In [ ]:
# 可视化计算量对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = [f"{r['pixel_dim']}→{r['latent_dim']}" for r in results]
pixel_times = [r["pixel_time_ms"] for r in results]
latent_times = [r["latent_time_ms"] for r in results]
speedups = [r["speedup"] for r in results]

x_pos = np.arange(len(results))
width = 0.35

bars1 = axes[0].bar(x_pos - width/2, pixel_times, width, color="tomato", label="像素空间", alpha=0.85)
bars2 = axes[0].bar(x_pos + width/2, latent_times, width, color="steelblue", label="隐空间", alpha=0.85)
axes[0].set_title("像素空间 vs 隐空间 单次扩散耗时对比", fontsize=13)
axes[0].set_ylabel("时间 (ms)")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f"维度{l}" for l in labels], fontsize=9)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(x_pos, speedups, color="mediumpurple", alpha=0.85, edgecolor="white")
axes[1].set_title("隐空间相对像素空间的加速比", fontsize=13)
axes[1].set_ylabel("加速比 (x)")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f"压缩{r['compression']}x" for r in results], fontsize=10)
axes[1].grid(alpha=0.3)
for i, v in enumerate(speedups):
    axes[1].text(i, v + 0.1, f"{v:.1f}x", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("../docs/assets/18-compute-comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("图片已保存")


## Part 6：数学性质验证

验证 LDM 核心数学性质，确保实现的正确性。


In [ ]:
# ============================================================
# 性质验证（供 pytest 使用的核心逻辑）
# ============================================================

def verify_compression_ratio(ae, pixel_dim, latent_dim):
    """验证自编码器实际压缩比"""
    return pixel_dim / latent_dim

def verify_reconstruction_quality(ae, dataset, threshold=0.5):
    """验证重建误差在合理范围内"""
    mses = []
    for x in dataset[:20]:
        x_hat, _ = ae.reconstruct(x)
        mses.append(np.mean((x_hat - x) ** 2))
    return np.mean(mses) < threshold

def verify_text_conditioning(denoiser, text_dim, latent_dim, T):
    """验证不同文字条件产生不同结果"""
    z = np.random.randn(latent_dim)
    text_a = np.random.randn(text_dim)
    text_b = np.random.randn(text_dim)
    pred_a = denoiser.predict_noise(z, text_a, T // 2, T)
    pred_b = denoiser.predict_noise(z, text_b, T // 2, T)
    return np.linalg.norm(pred_a - pred_b) > 1e-6

def verify_ddim_determinism(denoiser, text_vec, alpha_bar, n_steps=10):
    """验证 DDIM 确定性采样"""
    np.random.seed(99)
    z1 = latent_ddim_sample(denoiser, text_vec, alpha_bar, n_steps)
    np.random.seed(99)
    z2 = latent_ddim_sample(denoiser, text_vec, alpha_bar, n_steps)
    return np.allclose(z1, z2)

def verify_forward_noise_level(alpha_bar, T):
    """验证正向扩散：t=T 时的信号占比显著低于 t=0"""
    # alpha_bar 单调递减：t=T 时噪声远多于信号
    # 验证：alpha_bar[-1] < alpha_bar[0] 的一半
    return alpha_bar[-1] < alpha_bar[0] * 0.5

# 运行所有验证
ratio = verify_compression_ratio(ae, PIXEL_DIM, LATENT_DIM)
recon_ok = verify_reconstruction_quality(ae, dataset)
cond_ok = verify_text_conditioning(denoiser, TEXT_DIM, LATENT_DIM, T)
deter_ok = verify_ddim_determinism(denoiser, text_cat, alpha_bar)
noise_ok = verify_forward_noise_level(alpha_bar, T)

print("=== 数学性质验证 ===")
print(f"  压缩比正确（{PIXEL_DIM}/{LATENT_DIM} = {ratio:.0f}x）: {'✓' if ratio == 16 else '✗'}")
print(f"  重建误差在合理范围内（< 0.5）: {'✓' if recon_ok else '✗'}")
print(f"  文字条件产生不同预测: {'✓' if cond_ok else '✗'}")
print(f"  DDIM 确定性采样（相同种子结果相同）: {'✓' if deter_ok else '✗'}")
print(f"  正向扩散 t=T 时噪声水平足够高: {'✓' if noise_ok else '✗'}")

all_pass = ratio == 16 and recon_ok and cond_ok and deter_ok and noise_ok
print(f"\n全部验证: {'✓ PASS' if all_pass else '✗ FAIL'}")
assert all_pass, "存在未通过的验证！"


## 总结

本 notebook 演示了 Latent Diffusion Models 的三个核心机制：

| 机制 | 作用 | 关键数字 |
|------|------|---------|
| 自编码器 | 压缩图片到隐空间 | 16x 压缩 |
| 条件去噪 | 文字控制生成方向 | Cross-Attention 简化版 |
| 隐空间 DDIM | 快速确定性采样 | 20步即可收敛 |

**核心洞察**：在隐空间做扩散，计算量减少 ~100x，同时保持生成质量。这就是 Stable Diffusion 能跑在普通 GPU 上的原因。

---

论文：Rombach et al., "High-Resolution Image Synthesis with Latent Diffusion Models", CVPR 2022, [arXiv:2112.10752](https://arxiv.org/abs/2112.10752)
